In [10]:
DATASET_FOLDERS = [
                  #  '../../../../../data/datasets/msk_mri_h5/varnet_msk_dataset2/multicoil_test',
                  #  '../../../../../data/datasets/msk_mri_h5/varnet_msk_dataset2/multicoil_train',
                  #  '../../../../../data/datasets/msk_mri_h5/varnet_msk_dataset2/multicoil_val',
                   '../../../../../data/datasets/msk_mri_batch_2_samples_h5/varnet_mrpro'
                   ]
import os
import h5py
import random

folder = random.choice(DATASET_FOLDERS)
files = [f for f in os.listdir(folder) if f.endswith('.h5')]
random_file = os.path.join(folder, random.choice(files))
print(random_file)

../../../../../data/datasets/msk_mri_batch_2_samples_h5/varnet_mrpro/meas_MID00057_FID141038_COR_PD_FS.h5


In [11]:
# Pretty-print ISMRMRD header and show dataset shapes (robust formatting, plain text output)
import xml.dom.minidom as minidom
import json
import h5py
import numpy as np
import xml.etree.ElementTree as ET


def _strip_xml_namespaces(xml_str: str) -> str:
    try:
        root = ET.fromstring(xml_str)
    except Exception:
        return xml_str
    for elem in root.iter():
        # Strip namespace from tag
        if isinstance(elem.tag, str) and '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]
        # Strip namespace from attribute names
        if elem.attrib:
            new_attrib = {}
            for k, v in elem.attrib.items():
                if '}' in k:
                    k = k.split('}', 1)[1]
                new_attrib[k] = v
            elem.attrib.clear()
            elem.attrib.update(new_attrib)
    return ET.tostring(root, encoding='utf-8').decode('utf-8')


def _pretty_xml(s: str) -> str | None:
    try:
        # Remove namespaces to avoid ns0 prefixes
        no_ns = _strip_xml_namespaces(s)
        root = ET.fromstring(no_ns)
        rough = ET.tostring(root, encoding="utf-8")
        pretty = minidom.parseString(rough).toprettyxml(indent="  ")
        pretty = "\n".join([line for line in pretty.splitlines() if line.strip()])
        return pretty
    except Exception:
        return None


def _display_block(title: str, body: str, lang: str = "xml") -> None:
    # Plain text output, no markdown
    print(title)
    print(body)


def _to_str_from_dataset(ds: h5py.Dataset) -> str:
    # Try asstr (available in h5py>=3) for string datasets
    try:
        return ds.asstr()[()].strip()
    except Exception:
        pass
    # Raw read
    obj = ds[()]
    # If bytes-like directly
    if isinstance(obj, (bytes, bytearray)):
        return bytes(obj).replace(b"\x00", b"").decode("utf-8", errors="replace").strip()
    # If numpy scalar string/bytes
    if np.isscalar(obj):
        if isinstance(obj.item(), (bytes, bytearray)):
            return obj.item().replace(b"\x00", b"").decode("utf-8", errors="replace").strip()
        try:
            return str(obj.item())
        except Exception:
            pass
    # If numpy array of uint8/bytes
    if isinstance(obj, np.ndarray):
        if obj.dtype == np.uint8:
            return bytes(obj.tobytes()).replace(b"\x00", b"").decode("utf-8", errors="replace").strip()
        if obj.dtype.kind == 'S':  # fixed-width bytes strings
            try:
                joined = b"".join(obj.tolist())
            except Exception:
                joined = bytes(obj.tobytes())
            return joined.replace(b"\x00", b"").decode("utf-8", errors="replace").strip()
        if obj.dtype.kind == 'U':  # unicode strings
            try:
                return "".join(obj.tolist()).strip()
            except Exception:
                return str(obj)
    # Fallback
    return str(obj)


with h5py.File(random_file, 'r') as f:
    # print keys
    print(list(f.keys()))

    # Decode header robustly to Python str
    header_str = _to_str_from_dataset(f["ismrmrd_header"]) or ""

    # Try pretty XML first
    printed = False
    pretty_xml = _pretty_xml(header_str)
    if pretty_xml:
        _display_block("ISMRMRD Header (XML)", pretty_xml, lang="xml")
        printed = True

    # If not XML or parsing failed, try JSON
    if not printed:
        try:
            obj = json.loads(header_str)
            _display_block("ISMRMRD Header (JSON)", json.dumps(obj, indent=2), lang="json")
            printed = True
        except Exception:
            pass

    # Fallback: raw text block
    if not printed:
        _display_block("ISMRMRD Header (raw)", header_str, lang="text")

    # Print dataset shapes
    print("kspace shape:", f["kspace"].shape)
    print("reconstruction_rss shape:", f["reconstruction_rss"].shape)

['ismrmrd_header', 'kspace', 'reconstruction_rss']
ISMRMRD Header (XML)
<?xml version="1.0" ?>
<ismrmrdHeader schemaLocation="http://www.ismrm.org/ISMRMRD ismrmrd.xsd">
  <subjectInformation>
    <patientName>xxxxxxxxxxxxxxx</patientName>
    <patientWeight_kg>66.6790009</patientWeight_kg>
    <patientHeight_m>1626</patientHeight_m>
    <patientID>18.0.273306737</patientID>
    <patientGender>F</patientGender>
  </subjectInformation>
  <studyInformation>
    <studyTime>08:33:00</studyTime>
    <studyID>273306745</studyID>
  </studyInformation>
  <measurementInformation>
    <measurementID>169597_273306737_273306745_57</measurementID>
    <patientPosition>FFS</patientPosition>
    <protocolName>COR PD FS</protocolName>
    <measurementDependency>
      <dependencyType>SenMap</dependencyType>
      <measurementID>169597_273306737_273306745_51</measurementID>
    </measurementDependency>
    <measurementDependency>
      <dependencyType>Noise</dependencyType>
      <measurementID>169597_2

In [9]:
# Dataset-wide statistics across all files and splits (plain text output)
import os
import json
import math
import h5py
import numpy as np
from collections import Counter, defaultdict
from statistics import mean
import xml.etree.ElementTree as ET


def _safe_mean(vals):
    vals = [v for v in vals if v is not None]
    return float(mean(vals)) if vals else float('nan')


def _extract_text(elem):
    return elem.text.strip() if elem is not None and elem.text is not None else None


def _strip_xml_ns_text(xml_str: str) -> ET.Element | None:
    try:
        root = ET.fromstring(xml_str)
    except Exception:
        return None
    for e in root.iter():
        if isinstance(e.tag, str) and '}' in e.tag:
            e.tag = e.tag.split('}', 1)[1]
        if e.attrib:
            e.attrib = { (k.split('}',1)[1] if '}' in k else k): v for k, v in e.attrib.items() }
    return root


def parse_header_fields(header_str: str):
    root = _strip_xml_ns_text(header_str)
    if root is None:
        return {
            'patient_id': None,
            'field_strength_T': None,
            'receiver_channels': None,
            'encoded_matrix': None,
            'recon_matrix': None,
        }
    # patient id (prefer subjectInformation/patientID, fallback to userParameters PatientLOID)
    patient_id = _extract_text(root.find('.//subjectInformation/patientID'))
    if not patient_id:
        # look for userParameterString with name == PatientLOID
        for ups in root.findall('.//userParameters/userParameterString'):
            name = _extract_text(ups.find('name'))
            if name == 'PatientLOID':
                patient_id = _extract_text(ups.find('value'))
                if patient_id:
                    break
    # field strength
    fs_txt = _extract_text(root.find('.//acquisitionSystemInformation/systemFieldStrength_T'))
    field_strength_T = None
    if fs_txt:
        try:
            field_strength_T = float(fs_txt)
        except Exception:
            pass
    # receiver channels
    rc_txt = _extract_text(root.find('.//acquisitionSystemInformation/receiverChannels'))
    receiver_channels = None
    if rc_txt:
        try:
            receiver_channels = int(rc_txt)
        except Exception:
            # sometimes float-like text
            try:
                receiver_channels = int(float(rc_txt))
            except Exception:
                pass
    # matrix sizes
    def _matrix(node_path):
        n = root.find(node_path)
        if n is None:
            return None
        x = _extract_text(n.find('x'))
        y = _extract_text(n.find('y'))
        z = _extract_text(n.find('z'))
        try:
            return (int(x), int(y), int(z))
        except Exception:
            try:
                return (int(float(x)), int(float(y)), int(float(z)))
            except Exception:
                return None
    encoded_matrix = _matrix('.//encoding/encodedSpace/matrixSize')
    recon_matrix = _matrix('.//encoding/reconSpace/matrixSize')
    return {
        'patient_id': patient_id,
        'field_strength_T': field_strength_T,
        'receiver_channels': receiver_channels,
        'encoded_matrix': encoded_matrix,
        'recon_matrix': recon_matrix,
    }


def infer_split(path_str: str) -> str:
    p = path_str.lower()
    if 'train' in p:
        return 'train'
    if 'val' in p or 'valid' in p:
        return 'val'
    if 'test' in p:
        return 'test'
    return 'unknown'


def read_header_str_from_file(f: h5py.File) -> str:
    # Reuse helper from previous cell if available
    try:
        return _to_str_from_dataset(f['ismrmrd_header'])
    except NameError:
        pass
    # Fallback minimal decoder
    obj = f['ismrmrd_header'][()]
    if isinstance(obj, (bytes, bytearray)):
        return bytes(obj).replace(b'\x00', b'').decode('utf-8', errors='replace').strip()
    try:
        return str(obj)
    except Exception:
        return ''


all_files = []
for root_dir in DATASET_FOLDERS:
    if not os.path.isdir(root_dir):
        continue
    for name in os.listdir(root_dir):
        if name.endswith('.h5'):
            all_files.append(os.path.join(root_dir, name))

num_files = len(all_files)
if num_files == 0:
    print('No .h5 files found in DATASET_FOLDERS')
else:
    patients = set()
    field_strengths = []
    field_strength_counter = Counter()
    coils_list = []  # per file
    coils_counter = Counter()
    slices_per_file = []
    slices_by_split = defaultdict(list)
    coils_by_split = defaultdict(list)
    matrices_kspace = Counter()  # (H, W) from kspace
    matrices_encoded = Counter()  # (x,y,z)
    matrices_recon = Counter()  # (x,y,z)

    for fp in all_files:
        split = infer_split(fp)
        try:
            with h5py.File(fp, 'r') as f:
                # header
                header_str = read_header_str_from_file(f) or ''
                meta = parse_header_fields(header_str)
                if meta.get('patient_id'):
                    patients.add(meta['patient_id'])
                # field strength
                fs = meta.get('field_strength_T')
                if fs is not None:
                    field_strengths.append(fs)
                    field_strength_counter[round(fs, 3)] += 1
                # coils: prefer metadata else kspace dim
                rc = meta.get('receiver_channels')
                # shapes
                if 'kspace' in f:
                    ks_shape = f['kspace'].shape
                    # Expect (slices, coils, H, W) for multicoil
                    if len(ks_shape) >= 4:
                        slices = int(ks_shape[0])
                        coils_dim = int(ks_shape[1])
                        H, W = int(ks_shape[-2]), int(ks_shape[-1])
                        matrices_kspace[(H, W)] += 1
                    else:
                        # fallback generic
                        slices = int(ks_shape[0]) if len(ks_shape) > 0 else 0
                        coils_dim = int(ks_shape[1]) if len(ks_shape) > 1 else None
                    slices_per_file.append(slices)
                    slices_by_split[split].append(slices)
                    if rc is None and coils_dim is not None:
                        rc = coils_dim
                if rc is not None:
                    coils_list.append(rc)
                    coils_by_split[split].append(rc)
                    coils_counter[int(rc)] += 1
                # matrices from metadata
                if meta.get('encoded_matrix'):
                    matrices_encoded[meta['encoded_matrix']] += 1
                if meta.get('recon_matrix'):
                    matrices_recon[meta['recon_matrix']] += 1
        except Exception as e:
            # Skip unreadable files
            continue

    # Compute aggregates
    unique_patients = len(patients)
    total_slices = sum(slices_per_file) if slices_per_file else 0
    avg_slices = _safe_mean(slices_per_file)
    min_slices = min(slices_per_file) if slices_per_file else 0
    max_slices = max(slices_per_file) if slices_per_file else 0
    min_coils = min(coils_list) if coils_list else None
    max_coils = max(coils_list) if coils_list else None

    print('Dataset summary')
    print(f'- Files: {num_files}')
    print(f'- Unique patients: {unique_patients}')

    print('\nField strength (Tesla) distribution')
    if field_strength_counter:
        for val, cnt in sorted(field_strength_counter.items(), key=lambda x: x[0]):
            print(f'- {val}: {cnt}')
    else:
        print('- (none)')

    print('\nSlices per file (overall)')
    print(f'- Total slices: {total_slices}')
    print(f'- Avg slices: {avg_slices:.2f}')
    print(f'- Min slices: {min_slices}')
    print(f'- Max slices: {max_slices}')

    # Per split slice stats
    for sp in ['train', 'val', 'test', 'unknown']:
        svals = slices_by_split.get(sp, [])
        if not svals:
            continue
        print(f'- {sp}: n={len(svals)}, total={sum(svals)}, avg={_safe_mean(svals):.2f}, min={min(svals)}, max={max(svals)}')

    print('\nReceiver channels (per scan)')
    if coils_counter:
        print(f'- Min coils: {min_coils}')
        print(f'- Max coils: {max_coils}')
        for k, v in sorted(coils_counter.items()):
            print(f'- {k}: {v}')
    else:
        print('- (none)')

    print('\nMatrix sizes')
    if matrices_kspace:
        for (H, W), v in sorted(matrices_kspace.items()):
            print(f'- kspace HxW {H}x{W}: {v}')
    else:
        print('- kspace: (none)')
    if matrices_encoded:
        for (x, y, z), v in sorted(matrices_encoded.items()):
            print(f'- encodedSpace {x}x{y}x{z}: {v}')
    if matrices_recon:
        for (x, y, z), v in sorted(matrices_recon.items()):
            print(f'- reconSpace {x}x{y}x{z}: {v}')

Dataset summary
- Files: 2653
- Unique patients: 450

Field strength (Tesla) distribution
- 1.494: 2653

Slices per file (overall)
- Total slices: 79582
- Avg slices: 30.00
- Min slices: 12
- Max slices: 80
- train: n=1855, total=55661, avg=30.01, min=12, max=80
- val: n=398, total=12027, avg=30.22, min=12, max=72
- test: n=400, total=11894, avg=29.73, min=12, max=72

Receiver channels (per scan)
- Min coils: 4
- Max coils: 46
- 4: 41
- 8: 169
- 10: 6
- 12: 698
- 14: 34
- 15: 362
- 16: 1056
- 18: 12
- 20: 50
- 24: 25
- 26: 50
- 30: 99
- 32: 4
- 34: 28
- 36: 6
- 38: 2
- 40: 7
- 42: 1
- 44: 1
- 46: 2

Matrix sizes
- kspace HxW 250x640: 1
- kspace HxW 270x768: 1
- kspace HxW 312x512: 1
- kspace HxW 320x640: 20
- kspace HxW 320x768: 3
- kspace HxW 324x512: 2
- kspace HxW 336x512: 2
- kspace HxW 336x640: 3
- kspace HxW 340x512: 5
- kspace HxW 340x640: 1
- kspace HxW 350x896: 2
- kspace HxW 352x640: 6
- kspace HxW 354x640: 1
- kspace HxW 356x640: 1
- kspace HxW 360x512: 14
- kspace HxW 362x7